# Expérience 1 — Combien coûte la darija en tokens ?

**Question :** pour une même phrase, combien de tokens en plus la darija coûte-t-elle par rapport au français et à l'arabe standard, selon le tokenizer ?

**Métriques :**
- **fertilité** = nombre moyen de tokens par mot ;
- **ratio de parité** = tokens de la phrase en darija / tokens de la même phrase en français (Petrov et al., NeurIPS 2023) ;
- **taux d'octets** = part des tokens qui ne forment pas un caractère complet (lettre arabe coupée en octets) ou inconnus.

**Données :**
- **FLORES+** (dès maintenant) : français, arabe standard, darija marocaine, darija tunisienne. Pas d'algérien.
- **MADAR** (dès que l'accès est obtenu) : ajoute Alger, Rabat, Fès, Tunis, Sfax.

**Résultat (à remplir à la fin) :**

## 0. Mise en place

Sur Colab : *Exécution → Modifier le type d'exécution* n'est pas nécessaire, **le CPU suffit** pour cette expérience.

Avant la première exécution, ajoutez deux secrets dans Colab (icône 🔑 à gauche) :
- `HF_TOKEN` : votre token Hugging Face (huggingface.co → Settings → Access Tokens, type *Read*) ;
- `GITHUB_TOKEN` : un token GitHub avec le droit `repo` (seulement si le dépôt est privé).

Pour **Llama 3** et **Gemma**, il faut aussi accepter une fois leurs conditions sur leur page Hugging Face (bouton *Agree*). Pour **FLORES+**, idem sur la page du dataset `openlanguagedata/flores_plus`.

In [ ]:
import os, sys

REPO = "Ninessss/darija-llm"
BRANCH = "exp1/cout-tokens"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    if not os.path.exists("/content/darija-llm"):
        try:
            gh = userdata.get("GITHUB_TOKEN")
            url = f"https://{gh}@github.com/{REPO}.git"
        except Exception:
            url = f"https://github.com/{REPO}.git"
        !git clone -q -b {BRANCH} {url} /content/darija-llm
    %cd /content/darija-llm
    !git checkout -q {BRANCH} && git pull -q   # récupère la dernière version, même si le dépôt était déjà cloné
    !echo "Commit utilisé : $(git rev-parse --short HEAD)"
    !pip install -q transformers tokenizers datasets sentencepiece protobuf pandas matplotlib
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

sys.path.insert(0, os.getcwd())
print("Dossier de travail :", os.getcwd())

In [ ]:
from huggingface_hub import login

try:
    token = userdata.get("HF_TOKEN") if IN_COLAB else os.environ.get("HF_TOKEN")
    login(token=token)
    print("Connecté à Hugging Face")
except Exception as e:
    print("Pas de token Hugging Face : Llama 3, Gemma et FLORES+ ne pourront pas être chargés.", e)

## 1. Les tokenizers comparés

Les identifiants marqués `# à vérifier` sont ceux dont le nom exact sur Hugging Face doit être confirmé. Si un tokenizer ne se charge pas, la cellule l'indique et continue avec les autres.

In [ ]:
from transformers import AutoTokenizer

TOKENIZER_IDS = {
    "GPT-2":      "gpt2",
    "Llama 3":    "meta-llama/Meta-Llama-3-8B",       # accès à demander
    "Qwen 2.5":   "Qwen/Qwen2.5-0.5B",
    "Gemma 2":    "google/gemma-2-2b",                # accès à demander
    "mBERT":      "bert-base-multilingual-cased",
    "DziriBERT":  "alger-ia/dziribert",
    "DarijaBERT": "SI2M-Lab/DarijaBERT",
    # TunBERT écarté : le tokenizer publié sur tunis-ai/TunBERT a le vocabulaire du BERT
    # anglais (30 522 tokens) et coupe l'arabe lettre par lettre. Ce n'est pas un vrai
    # tokenizer tunisien, l'inclure fausserait la comparaison (à signaler dans le rapport).
}

tokenizers = {}
for name, hf_id in TOKENIZER_IDS.items():
    try:
        tokenizers[name] = AutoTokenizer.from_pretrained(hf_id)
        print(f"OK      {name:<11} {hf_id}  (vocabulaire : {len(tokenizers[name]):,})")
    except Exception as e:
        print(f"ÉCHEC   {name:<11} {hf_id}  -> {type(e).__name__}: {str(e)[:100]}")

## 2. Les phrases parallèles

FLORES+ contient les **mêmes phrases** traduites dans chaque langue, alignées par leur `id`. On garde le split `dev` (997 phrases) et `devtest` (1 012 phrases).

In [ ]:
import pandas as pd
from datasets import load_dataset

FLORES_LANGS = {
    "Français":         "fra_Latn",
    "Arabe standard":   "arb_Arab",
    "Darija marocaine": "ary_Arab",
    "Darija tunisienne":"aeb_Arab",
}

def load_flores(code):
    parts = []
    for split in ["dev", "devtest"]:
        ds = load_dataset("openlanguagedata/flores_plus", code, split=split)
        parts.append(ds.to_pandas()[["id", "text"]].assign(split=split))
    return pd.concat(parts).set_index(["split", "id"])["text"]

flores = pd.DataFrame({lang: load_flores(code) for lang, code in FLORES_LANGS.items()}).dropna()
print(f"{len(flores)} phrases alignées dans les {flores.shape[1]} langues")
flores.head(3)

**Par défaut, on mesure le texte brut**, tel que les gens l'écrivent : c'est ce que reçoit réellement une IA. La normalisation (variantes d'alef, ة → ه…) modifie le texte et fausserait la mesure du coût réel.

> Variante possible : `NORMALIZE = True` applique `src/normalize.py`. Attention, la version actuelle de `normalize` prend le français pour de l'arabizi et le met en minuscules (bug connu, à corriger avant l'expérience 2).

In [ ]:
from src.normalize import normalize

NORMALIZE = False
data = flores.map(normalize) if NORMALIZE else flores.copy()
data.head(3)

## 3. Les mesures

Pour chaque tokenizer et chaque langue :
- `tokens_total` : nombre total de tokens sur tout le corpus ;
- `fertilite` : tokens par mot ;
- `parite_vs_fr` : moyenne, phrase par phrase, de tokens(langue) / tokens(français) ;
- `parite_vs_msa` : la même chose par rapport à l'arabe standard ;
- `taux_octets` : part des tokens inconnus ou coupés en octets.

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from src.metrics import is_broken_token

def count_tokens(tok, texts):
    return np.array([len(tok.encode(t, add_special_tokens=False)) for t in texts])

rows = []
for tok_name, tok in tqdm(tokenizers.items()):
    counts = {lang: count_tokens(tok, data[lang]) for lang in data.columns}
    for lang in data.columns:
        texts = data[lang]
        n_words = texts.str.split().str.len().to_numpy()
        ids = [i for t in texts for i in tok.encode(t, add_special_tokens=False)]
        rows.append({
            "tokenizer": tok_name,
            "langue": lang,
            "tokens_total": int(counts[lang].sum()),
            "fertilite": counts[lang].sum() / n_words.sum(),
            "parite_vs_fr": float(np.mean(counts[lang] / np.maximum(counts["Français"], 1))),
            "parite_vs_msa": float(np.mean(counts[lang] / np.maximum(counts["Arabe standard"], 1))),
            "taux_octets": float(np.mean([is_broken_token(tok, i) for i in ids])),
        })

results = pd.DataFrame(rows)
os.makedirs("results/figures", exist_ok=True)
results.to_csv("results/exp1_flores.csv", index=False)
results.round(3)

In [ ]:
# Le tableau principal : ratio de parité par rapport au français
parite = results.pivot(index="tokenizer", columns="langue", values="parite_vs_fr")[list(FLORES_LANGS)]
parite = parite.sort_values("Darija marocaine", ascending=False)
parite.round(2)

In [ ]:
results.pivot(index="tokenizer", columns="langue", values="taux_octets")[list(FLORES_LANGS)].mul(100).round(1)

> **Piège à surveiller.** Pour les tokenizers de type BERT (mBERT, DziriBERT…), un mot inconnu devient un seul token `[UNK]`. Le ratio de parité peut alors paraître **bon** (peu de tokens) alors que le texte est en réalité **perdu**. Il faut toujours lire la parité **avec** le taux d'octets/inconnus : une parité basse avec un taux élevé est un mauvais résultat, pas un bon.

## 4. Figure : la heatmap des ratios de parité

Une case à **2,0** veut dire : la même phrase coûte **deux fois plus de tokens** que le français avec ce tokenizer.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 0.5 * len(parite) + 1.5))
im = ax.imshow(parite.values, cmap="Reds", vmin=1, aspect="auto")
ax.set_xticks(range(parite.shape[1]), parite.columns, rotation=20, ha="right")
ax.set_yticks(range(parite.shape[0]), parite.index)
for i in range(parite.shape[0]):
    for j in range(parite.shape[1]):
        v = parite.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if v > parite.values.max() * 0.7 else "black")
ax.set_title("Ratio de parité (tokens / tokens du français), FLORES+")
fig.colorbar(im, ax=ax, label="× tokens du français")
fig.tight_layout()
fig.savefig("results/figures/exp1_parite_flores.png", dpi=200)
plt.show()

## 5. Voir le découpage : une phrase, plusieurs tokenizers

Chaque couleur = un token. Un **�** signale une lettre coupée en octets : le tokenizer ne la connaît pas comme un tout.

C'est cette visualisation qui servira d'ouverture à l'oral.

In [ ]:
from IPython.display import HTML, display

COLORS = ["#fde2e4", "#e2ece9", "#dfe7fd", "#fff1c1", "#e8dff5", "#d7f2f5"]

def show_tokens(text, names=None):
    names = names or list(tokenizers)
    rtl = any("\u0600" <= c <= "\u06FF" for c in text)
    html = []
    for name in names:
        tok = tokenizers[name]
        ids = tok.encode(text, add_special_tokens=False)
        spans = "".join(
            f'<span style="background:{COLORS[k % len(COLORS)]};padding:2px 1px;margin:0 1px;border-radius:3px">'
            f'{tok.decode([i]).replace(" ", "&nbsp;")}</span>'
            for k, i in enumerate(ids)
        )
        html.append(
            f'<tr><td style="padding:4px 12px 4px 0;white-space:nowrap"><b>{name}</b><br>{len(ids)} tokens</td>'
            f'<td dir="{"rtl" if rtl else "ltr"}" style="font-size:18px;line-height:2">{spans}</td></tr>'
        )
    display(HTML(f"<table>{''.join(html)}</table>"))

i = 0  # changez l'indice pour voir d'autres phrases
for lang in ["Français", "Darija marocaine"]:
    print(f"--- {lang}")
    show_tokens(data[lang].iloc[i])

## 6. Arabizi : la darija en lettres latines

Beaucoup de Marocains écrivent la darija en **lettres latines et chiffres** : `3` pour ع, `7` pour ح, `9` pour ق. Par exemple « ma 3reftch » pour « ما عرفتش ».

**Question :** pour la **même phrase**, l'arabizi coûte-t-il plus ou moins de tokens que l'alphabet arabe ?

**Deux effets s'opposent :**
- en lettres latines, chaque lettre ne fait qu'**un octet** (contre deux en arabe), donc les tokenizers byte-level devraient moins casser les lettres ;
- mais l'arabizi écrit les voyelles, n'a pas d'orthographe fixe et mélange **lettres et chiffres** (`m3a`, `7na`), ce que les tokenizers coupent souvent mal.

**Données :** le *Darija Open Dataset* (DODa), où chaque phrase marocaine existe en **arabizi**, en **alphabet arabe** et en **anglais**. Licence : libre pour un usage de recherche et d'enseignement.

In [ ]:
DODA_URL = "https://raw.githubusercontent.com/darija-open-dataset/dataset/main/sentences/sentences.csv"
doda_raw = pd.read_csv(DODA_URL)
print("Colonnes :", doda_raw.columns.tolist(), "|", len(doda_raw), "lignes")

def arabic_share(col):
    s = doda_raw[col].astype(str)
    return s.map(lambda t: sum("\u0600" <= c <= "\u06FF" for c in t) / max(len(t), 1)).mean()

cols = list(doda_raw.columns)
COL_AR  = "darija_ar" if "darija_ar" in cols else max(cols, key=arabic_share)
COL_EN  = "eng" if "eng" in cols else next(c for c in cols if c.lower().startswith("en"))
COL_LAT = "darija" if "darija" in cols else next(c for c in cols if c not in (COL_AR, COL_EN))
print(f"Arabizi = {COL_LAT!r} | Alphabet arabe = {COL_AR!r} | Anglais = {COL_EN!r}")

doda = (doda_raw[[COL_LAT, COL_AR, COL_EN]]
        .rename(columns={COL_LAT: "Arabizi", COL_AR: "Darija (alphabet arabe)", COL_EN: "Anglais"})
        .dropna().astype(str))
doda = doda[(doda.apply(lambda c: c.str.strip().str.len()) > 0).all(axis=1)].drop_duplicates()
print(len(doda), "phrases dans les 3 versions")
doda.sample(5, random_state=1)

Les mesures sont les mêmes qu'en section 3, mais la **référence** est l'anglais (DODa n'a pas de français). La colonne qui compte le plus est **`arabizi_vs_arabe`** : tokens en arabizi / tokens en alphabet arabe, pour la même phrase. Au-dessus de 1, l'arabizi coûte plus cher.

On ajoute `tokens_avec_chiffre` : la part des tokens qui contiennent un chiffre, pour voir si les `3`, `7`, `9` de l'arabizi sont coupés à part.

In [ ]:
rows_az = []
for tok_name, tok in tqdm(tokenizers.items()):
    counts = {v: count_tokens(tok, doda[v]) for v in doda.columns}
    for v in doda.columns:
        ids = [i for t in doda[v] for i in tok.encode(t, add_special_tokens=False)]
        pieces = [tok.decode([i]) for i in ids]
        rows_az.append({
            "tokenizer": tok_name,
            "version": v,
            "fertilite": counts[v].sum() / doda[v].str.split().str.len().sum(),
            "parite_vs_anglais": float(np.mean(counts[v] / np.maximum(counts["Anglais"], 1))),
            "arabizi_vs_arabe": float(np.mean(counts[v] / np.maximum(counts["Darija (alphabet arabe)"], 1))),
            "taux_octets": float(np.mean([is_broken_token(tok, i) for i in ids])),
            "tokens_avec_chiffre": float(np.mean([any(c.isdigit() for c in p) for p in pieces])),
        })

results_az = pd.DataFrame(rows_az)
results_az.to_csv("results/exp1_arabizi_doda.csv", index=False)

resume = results_az[results_az.version == "Arabizi"].set_index("tokenizer")[
    ["arabizi_vs_arabe", "parite_vs_anglais", "fertilite", "taux_octets", "tokens_avec_chiffre"]]
resume.sort_values("arabizi_vs_arabe", ascending=False).round(3)

In [ ]:
# Figure : coût par rapport à l'anglais, alphabet arabe contre arabizi
pv = results_az.pivot(index="tokenizer", columns="version", values="parite_vs_anglais")
pv = pv[["Darija (alphabet arabe)", "Arabizi"]].sort_values("Arabizi", ascending=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(pv)); w = 0.38
b1 = ax.bar(x - w/2, pv["Darija (alphabet arabe)"], w, label="Darija, alphabet arabe", color="#4C78A8")
b2 = ax.bar(x + w/2, pv["Arabizi"], w, label="Darija, arabizi", color="#F58518")
ax.axhline(1, color="grey", lw=1, ls="--")
ax.bar_label(b1, fmt="%.2f", fontsize=8); ax.bar_label(b2, fmt="%.2f", fontsize=8)
ax.set_xticks(x, pv.index, rotation=20, ha="right")
ax.set_ylabel("× tokens de l'anglais")
ax.set_title("Même phrase marocaine : alphabet arabe ou arabizi ? (DODa)")
ax.legend()
fig.tight_layout()
fig.savefig("results/figures/exp1_arabizi_doda.png", dpi=200)
plt.show()

In [ ]:
# Une même phrase dans les deux alphabets
ex = doda.iloc[0]
for v in ["Arabizi", "Darija (alphabet arabe)", "Anglais"]:
    print(f"--- {v} : {ex[v]}")
    show_tokens(ex[v])

### 6 bis. Arabizi sur les phrases FLORES+ (transcription par un natif)

DODa contient surtout des phrases courtes et quotidiennes. Pour comparer l'arabizi **au français** sur les mêmes phrases que la section 3, il faut des phrases FLORES+ écrites en arabizi. Aucun dataset ne les fournit : un natif doit les **transcrire à la main**.

1. La cellule suivante crée `data/manual/arabizi_flores_ma.csv` avec 50 phrases FLORES+ en darija marocaine.
2. Téléchargez-le, remplissez la colonne `arabizi` comme vous l'écririez sur WhatsApp, puis remettez-le au même endroit (dans le dépôt, avec un commit).
3. Relancez : la cellule d'après calcule la parité de l'arabizi par rapport au français.

In [ ]:
MANUAL = "data/manual/arabizi_flores_ma.csv"
os.makedirs("data/manual", exist_ok=True)
if not os.path.exists(MANUAL):
    tpl = flores[["Français", "Darija marocaine"]].iloc[:50].reset_index()
    tpl["arabizi"] = ""
    tpl.to_csv(MANUAL, index=False, encoding="utf-8")
    print("Modèle créé :", MANUAL, "-> à remplir")
else:
    print("Fichier déjà présent :", MANUAL)

In [ ]:
man = pd.read_csv(MANUAL).fillna("")
man = man[man["arabizi"].str.strip() != ""]
if len(man) < 10:
    print(f"Seulement {len(man)} phrases remplies : remplissez au moins 30 à 50 phrases.")
else:
    rows_m = []
    for tok_name, tok in tokenizers.items():
        fr = count_tokens(tok, man["Français"])
        for v in ["Darija marocaine", "arabizi"]:
            c = count_tokens(tok, man[v])
            rows_m.append({"tokenizer": tok_name, "version": v,
                           "parite_vs_fr": float(np.mean(c / np.maximum(fr, 1)))})
    res_m = pd.DataFrame(rows_m).pivot(index="tokenizer", columns="version", values="parite_vs_fr")
    res_m.to_csv("results/exp1_arabizi_flores_manuel.csv")
    print(f"{len(man)} phrases transcrites")
    display(res_m.round(2))

## 7. À noter dans `results/journal.md`

La cellule suivante prépare les lignes à copier dans le journal des résultats.

In [ ]:
import datetime, subprocess

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
today = datetime.date.today().isoformat()
for tok_name in parite.index:
    for lang in ["Darija marocaine", "Darija tunisienne"]:
        print(f"| {today} | 1 | {tok_name}, FLORES+, normalise={NORMALIZE} | parité vs fr ({lang}) | {parite.loc[tok_name, lang]:.2f} | — | {commit} | Ammar |")

## 8. Suite

- [ ] Ajouter **MADAR** dès que l'accès est obtenu : c'est la seule source avec l'**algérien**.
- [ ] Remplir les 50 phrases arabizi de la section 6 bis (darija marocaine) ; idéalement un natif tunisien et un algérien font de même.
- [ ] Corriger `detect_script` dans `src/normalize.py` (le français est pris pour de l'arabizi).
- [ ] Lire les premiers résultats : quel tokenizer pénalise le plus la darija ? L'arabe standard est-il mieux traité que la darija ?